#### Transform Orders Data - Explode Arrays
1. Access elements from JSON object
2. Deduplicate Array Elements
3. Exlpode Arrays
4. Write transformed data to silver schema

In [0]:
df_orders = spark.table('gizmobox_catalog_subbu.silver.py_orders_json')
display(df_orders)

##### 1. Access elements from JSON object

In [0]:
from pyspark.sql import functions as F

In [0]:
df_orders_normalized=(
                df_orders
                .select(
                    F.col("json_value.order_id").alias("order_id"),
                    F.col("json_value.order_date").alias("order_date"),
                    F.col("json_value.total_amount").alias("total_amount"),
                    df_orders.json_value.order_status.alias("order_status"),
                    "json_value.payment_method",
                    "json_value.customer_id",
                    "json_value.transaction_timestamp",
                    "json_value.items"
                )
)
display(df_orders_normalized)


##### 2. Deduplicate Array Elements

In [0]:
df_orders_normalized=(
                df_orders
                .select(
                    F.col("json_value.order_id").alias("order_id"),
                    F.col("json_value.order_date").alias("order_date"),
                    F.col("json_value.total_amount").alias("total_amount"),
                    "json_value.order_status",
                    "json_value.payment_method",
                    "json_value.customer_id",
                    "json_value.transaction_timestamp",
                    F.array_distinct("json_value.items").alias("items")
                )
)
display(df_orders_normalized)

##### 3. Exlpode Arrays

In [0]:
df_order_explode =(
                df_orders_normalized
                .select(
                    "order_id",
                    "order_date",
                    "total_amount",
                    "order_status",
                    "payment_method",
                    "customer_id",
                    "transaction_timestamp",
                    F.explode("items").alias("item")
                )
)
display(df_order_explode)



In [0]:
df_order_items =(
            df_order_explode
            .select(
                "order_id",
                "order_date",
                "total_amount",
                "order_status",
                "payment_method",
                "customer_id",
                "transaction_timestamp",
                "item.item_id",
                "item.name",
                "item.category",
                "item.price",
                "item.quantity",
                "item.details.brand",
                "item.details.color"
            )
)
display(df_order_items)

##### 4. Write transformed data to silver schema

In [0]:
df_order_items.writeTo('gizmobox_catalog_subbu.silver.py_orders').createOrReplace()

In [0]:
%sql
select * from gizmobox_catalog_subbu.silver.py_orders